SQL modelling

In [0]:
%sql
SELECT
    s.id                       AS id,
    s.name                     AS show_name,
    s.language                 AS language,
    s.genres                   AS genre,
    e.id                       AS episode_id,
    e.name                     AS episode_name,
    e.season                   AS season,
    e.airdate                  AS airdate,
    e.runtime                  AS runtime,
    c.person_name              AS cast_name,
    c.character_name           AS character_name
FROM silver.shows s
INNER JOIN silver.episodes e
    ON s.id = e.show_id
LEFT JOIN silver.cast c
    ON s.id = c.show_id;

o	A query with window functions

In [0]:
%sql
SELECT
      DISTINCT  name,
        language
        
    FROM 
   ( 
SELECT
  id,
        name,
        language,
        DENSE_RANK() OVER (
            PARTITION BY id
            ORDER BY runtime DESC
        ) AS id_rank
    FROM silver.shows)
WHERE id_rank = 1

o	A query with joins and aggregations

In [0]:
%sql

CREATE OR REPLACE TABLE cast_count
USING DELTA
AS
SELECT
    show_id,
    person_name,
    COUNT(person_id) AS appearances
FROM silver.cast
GROUP BY
    show_id,
    person_name;


In [0]:
%sql
SELECT
    show_id AS id,
    person_name,
    appearances,
    cast_rank,
    sum(appearances),
    count(show_id) as total
FROM (
    SELECT
        show_id,
        person_name,
        appearances,
        DENSE_RANK() OVER (
            PARTITION BY show_id
            ORDER BY appearances DESC
        ) AS cast_rank
    FROM cast_count
) ranked_cast
WHERE cast_rank <= 3
GROUP by    
    show_id,
    person_name,
    appearances,
    cast_rank

Fastest & Cleanest (No redundant aggregation)

In [0]:

%sql
SELECT
    show_id     AS id,
    person_name,
    COUNT(person_id) AS appearances
    
FROM  silver.cast
GROUP BY   show_id ,
    person_name
HAVING COUNT(person_id) <= 3




o	Why certain predicates filter early

In [0]:
%sql
SELECT
    s.id                       AS id,
    s.name                     AS show_name,
    s.language                 AS language,
    s.genres                   AS genre,
    e.id                       AS episode_id,
    e.name                     AS episode_name,
    e.season                   AS season,
    e.airdate                  AS airdate,
    e.runtime                  AS runtime,
    c.person_name              AS cast_name,
    c.character_name           AS character_name
FROM silver.shows  s

INNER JOIN silver.episodes e
    ON s.id = e.show_id
LEFT JOIN silver.cast c
    ON s.id = c.show_id;

In [0]:

%sql
WITH filtered_shows AS (
    SELECT
        id,
        name,
        language,
        genres
    FROM silver.shows
    WHERE language = 'English'
),
filtered_episodes AS (
    SELECT
        id,
        show_id,
        name,
        season,
        airdate,
        runtime
    FROM silver.episodes
    WHERE season IS NOT NULL
)
SELECT
    s.id                 AS id,
    s.name               AS show_name,
    s.language            AS language,
    s.genres              AS genre,
    e.id                  AS episode_id,
    e.name                AS episode_name,
    e.season              AS season,
    e.airdate             AS airdate,
    e.runtime             AS runtime,
    c.person_name         AS cast_name,
    c.character_name      AS character_name
FROM filtered_shows s
INNER JOIN filtered_episodes e
    ON s.id = e.show_id
LEFT JOIN silver.cast c
    ON s.id = c.show_id;
